# GeoSR-4 — DINOv3 sat493m perceptual-backbone run (Colab GPU)

Standalone notebook -- only the DINOv3 sat493m run (D042), no DINOv2 section, so this can run in a fresh Colab session/account without repeating the DINOv2 run first.

`facebook/dinov3-vitl16-pretrain-sat493m` is ViT-L (300M params), satellite-pretrained (SAT-493M dataset) -- gated access, approved for this project already (D042). `--batch-size 4` is a conservative starting guess against T4's 15GB (this model is much bigger than DINOv2-small's 21M params) -- reduce further if it OOMs, see D024/D025 for the same kind of VGG-perceptual-loss OOM fix pattern.

Config otherwise matches D028/D029's VGG "quality run" protocol as closely as possible (30 epochs, lr 1e-4, lambda-perceptual 0.01, ICNR on) so eval numbers stay comparable to D028 (VGG) and the DINOv2 run already done separately.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
`transformers` is needed to load the DINOv3 backbone.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision transformers

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Hugging Face login (required -- gated model)
This opens an interactive prompt -- paste your HF **token** there (never in a code cell/chat). Must be logged in as the account that was granted access to `facebook/dinov3-vitl16-pretrain-sat493m` -- if you have multiple HF accounts, generate the token from **that specific account's** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (Read scope is enough), rather than using the device-code login flow (which can silently log you into the wrong account if your browser has another HF session active).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 4. Train: DINOv3 sat493m perceptual backbone

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir experiments/swinir_dino3_sat_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino3_sat_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Download the checkpoint

In [ ]:
from google.colab import files
files.download('experiments/swinir_dino3_sat_quality/swinir_epoch29.pt')